# Lab Day 2 — DeepWeeds: backbone, công thức huấn luyện, suy luận

**Võ Minh Quân · 2A202602429**

Notebook chạy toàn bộ bài lab trên Google Colab bằng **Runtime → Run all**.

| Bước | Ô | Nội dung | Tập dùng |
|---|---|---|---|
| 0 | 4 | Kiểm tra chia dữ liệu, EDA, loss ban đầu ≈ ln 9, overfit 1 batch | train/val |
| 1 | 5 | 6 backbone, cùng công thức nền, chọn backbone theo quy tắc định trước | **val** |
| 2 | 6–7 | T00 (3 seed) + 14 ablation + kết hợp, chọn công thức | **val** |
| 3 | 8 | ≥ 4 phương pháp suy luận, độ trễ p50/p95/p99, chọn phương pháp | **val** |
| 4 | 9 | Chung kết F01 × 3 seed, **test chạy một lần mỗi seed**, `eval.py score/grade` | **test** |
| 5 | 10 | `results.xlsx`, bảng, gói kết quả để nộp | — |

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.
**Bị ngắt kết nối?** Chạy lại *Run all*: lần train đã xong được bỏ qua, lần đang dở tiếp tục từ epoch cuối
(mọi kết quả lưu trên Google Drive). Test của chung kết không bao giờ chạy lại.
Mọi lựa chọn (backbone, công thức, suy luận) dùng **quy tắc định trước trên val**, xem `code/experiments.py`
và `code/step3_inference.py`.

In [ ]:
# ===================== CẤU HÌNH =====================
SMOKE = False   # True = chạy thử nhanh (4 ảnh/lớp, 1 epoch) chỉ để kiểm tra code; kết quả KHÔNG dùng để nộp
REPO_URL = "https://github.com/vminhquan/K4-Track4-Day2-Deeplearning-Advance.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/lab_day2_work"   # nơi lưu runs/, curves/, predictions/, results/
WORKERS = 2                                          # Colab có 2 CPU

In [ ]:
# Ô 1 — Drive, code, thư viện, GPU
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

CODE = f"{REPO}/submissions/2A202602429_vo_minh_quan/code"
WORK = "/content/smoke" if SMOKE else DRIVE_DIR
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

!pip -q install timm fvcore openpyxl tabulate

import torch, timm, torchvision, platform
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU, rồi Run all lại"
print("GPU:", torch.cuda.get_device_name(0))
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__)
print("Thư mục làm việc:", WORK, "| SMOKE =", SMOKE)

In [ ]:
# Hàm chạy script: in log trực tiếp và DỪNG notebook nếu script lỗi (để Run all không đi tiếp với kết quả thiếu)
SKIP = ("HF_TOKEN", "check_worker_number_rationality", "suggested max number of worker")

def sh(*args):
    env = dict(os.environ, PYTHONWARNINGS="ignore", PYTHONUNBUFFERED="1")
    if SMOKE:
        env["LAB_SMOKE"] = "4"
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(s in line for s in SKIP):
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}): {' '.join(map(str, args))} — xem log phía trên")

In [ ]:
# Ô 2 — Dữ liệu: images.zip (Zenodo, kiểm tra MD5, lưu đệm trên Drive) + nhãn fold 0 (GitHub tác giả)
import hashlib, urllib.request, zipfile

DATA = f"{REPO}/data"
IMAGES = f"{DATA}/images"
os.makedirs(f"{DATA}/labels", exist_ok=True)
ZIP = f"{DRIVE_DIR}/images.zip"        # lưu đệm trên Drive để lần sau không phải tải lại
os.makedirs(DRIVE_DIR, exist_ok=True)

if not (os.path.isdir(IMAGES) and len(os.listdir(IMAGES)) == 17509):
    if not os.path.exists(ZIP):
        print("Tải images.zip (~490 MB)...")
        urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", ZIP)
    h = hashlib.md5()
    with open(ZIP, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()} — xoá {ZIP} rồi chạy lại"
    print("MD5 đúng. Giải nén vào ổ cục bộ...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(IMAGES)
print("Số ảnh:", len(os.listdir(IMAGES)))

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{DATA}/labels/{n}.csv"):
        urllib.request.urlretrieve(f"{BASE}/{n}.csv", f"{DATA}/labels/{n}.csv")
print(sorted(os.listdir(f"{DATA}/labels")))

In [ ]:
# Ô 3 — Test tự viết cho các phần dễ sai (focal γ=0 ≡ CE, CutMix λ theo diện tích, gộp BN, EMA, LR warmup+cosine...)
r = subprocess.run([sys.executable, "-m", "unittest", "test_code"], cwd=CODE, capture_output=True, text=True,
                   env=dict(os.environ, PYTHONWARNINGS="ignore"))
print(r.stderr[-600:])
assert r.returncode == 0, "Test thất bại"

## Bước 0 — Kiểm tra dữ liệu và pipeline (README 2.1, GUIDE 1.2–1.3)

In [ ]:
sh(f"{CODE}/step0_checks.py")
from IPython.display import Image, display
for f in ["eda_class_distribution.png", "aug_basic.png", "aug_cutmix.png", "overfit_one_batch.png"]:
    display(Image(f"figures/{f}", width=700))

## Bước 1 — So sánh 6 backbone (cùng công thức nền, seed 0, 10 epoch) và chọn backbone theo val

In [ ]:
sh(f"{CODE}/experiments.py", "--group", "B", "--preload")
sh(f"{CODE}/experiments.py", "--select-backbone")
import pandas as pd, json
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(pd.read_csv("results/backbones.csv")[["exp_id", "backbone", "weight_tag", "params_M", "gmacs", "val_macro_f1",
        "val_top1", "train_time_per_epoch_s", "lat_b1_p50_ms", "lat_b1_p95_ms"]])
print(json.load(open("results/selection.json"))["backbone_reason"])

## Bước 2 — Công thức huấn luyện: T00 (3 seed, đo nhiễu) + T01–T14 (mỗi lần khác T00 một yếu tố)

In [ ]:
sh(f"{CODE}/experiments.py", "--group", "T", "--preload")

### Kết hợp các yếu tố tốt (C01/C02) và chốt công thức theo val

In [ ]:
sh(f"{CODE}/experiments.py", "--group", "C", "--preload", "--select-recipe")
sel = json.load(open("results/selection.json"))
print("Std T00 qua 3 seed:", sel["t00_std_3seeds"], "| ngưỡng:", sel["combo_threshold"])
display(pd.Series(sel["gains_vs_T00"], name="Δ macro-F1 val vs T00").sort_values())
print("Kết hợp:", [c["desc"] for c in sel["combos"]])
print("CÔNG THỨC CHỐT:", sel["recipe_exp"], sel["recipe_overrides"])

## Bước 3 — Phương pháp suy luận trên val + độ trễ đo đúng cách

In [ ]:
if not os.path.exists("results/inference_val.csv") or "inference" not in json.load(open("results/selection.json")):
    sh(f"{CODE}/step3_inference.py", "--preload", "--workers", WORKERS)
else:
    print("Bước 3 đã xong, bỏ qua (xoá results/inference_val.csv nếu muốn chạy lại)")
display(pd.read_csv("results/inference_val.csv")[["exp_id", "method", "K", "macro_f1", "top1", "ece", "p50_ms",
        "p95_ms", "delta_f1_vs_I00", "cost_vs_I00"]])
display(Image("figures/inference_tradeoff.png", width=750))
print("PHƯƠNG PHÁP CHỐT:", json.load(open("results/selection.json"))["inference"])

## Bước 4 — Chung kết: F01 × 3 seed, **test một lần mỗi seed**, `eval.py score` + `grade`
Cấu hình đã khai báo trong `results/selection.json` trước khi mở test. Script tự bỏ qua seed đã chạy test.

In [ ]:
sh(f"{CODE}/step4_final.py", "--preload", "--workers", WORKERS)
print(open("results/grade.txt").read())
display(Image("figures/confusion_F01_seed0.png", width=600))

## Bước 5 — `results.xlsx`, bảng tổng hợp, gói kết quả để tải về

In [ ]:
sh(f"{CODE}/make_results.py")
for sheet in ["Summary", "Final"]:
    display(pd.read_excel("results.xlsx", sheet_name=sheet))

# Gói các thư mục cần nộp (KHÔNG gói runs/ vì chứa checkpoint lớn)
import zipfile, glob
zpath = f"{WORK}/submission_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["results.xlsx", "curves/*", "predictions/*", "results/**/*", "figures/*"]:
        for f in glob.glob(pattern, recursive=True):
            if os.path.isfile(f):
                z.write(f)
print("Đã gói:", zpath, f"({os.path.getsize(zpath) / 1e6:.1f} MB)")
print("Tải file này từ Google Drive (MyDrive/lab_day2_work/) rồi giải nén vào submissions/2A202602429_vo_minh_quan/")